<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table { margin:1.5em auto; text-align:left; }
.lx-table caption { caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center; }
.lx-figure { margin:1.5em auto; text-align:center; }
.lx-figure figcaption { font-size:0.9em; margin-top:0.6em; text-align:center; }
</style>

# Duckiedrone ToF Sensor Architecture

[Measuring ToF sensor accuracy](./2-measuring-tof-sensor-accuracy.ipynb) compared received ranges with an independent reference. Each reading passes through several components before appearing on a ROS 2 topic. Understanding their responsibilities helps explain the message fields, recognize unusable readings, and investigate a stream that is slower than expected or has stopped.

This notebook addresses the following questions:

- How does a reading travel from a ToF sensor to a ROS 2 subscriber?

- Which component determines its units, timestamp, and range limits?

- How does the same sensor supply information to the flight controller?

- What constrains the publication rate, and how are configuration changes applied?

## The big picture

On a physical Duckiedrone, the VL53L1X ToF sensor performs the optical measurement. A driver on the Raspberry Pi reads the result over I2C and publishes it through Duckietown's DTPS communication system. A separate bridge converts that data into a ROS 2 `sensor_msgs/msg/Range` message.

The bottom sensor's ROS 2 topic has two relevant consumers: a subscriber used for inspection or analysis, and MAVROS, which forwards distance information to the PX4 flight controller. [Figure 1](#figure-1) shows both paths.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; max-width:100%; overflow-x:auto; text-align:left;">
              I2C                        DTPS                      ROS 2
+------------+ 30 Hz +-------------------+ 30 Hz +-----------------+ 30 Hz +-----------------+
| ToF sensor |------>| ToF driver        |------>| ROS 2 bridge    |------>| Subscriber node |
|  (bottom)  |       | driver-tof-bottom |       | ros2-tof-bottom |       +-----------------+
+------------+       +-------------------+       +-----------------+
                                                        |
                                                        | ROS 2, 30 Hz
                                                        v
                                           +------------------------+
                                           | MAVROS2                |
                                           | ros2-mavros            |
                                           | distance_sensor plugin |
                                           +------------------------+
                                                        |
                                                        | MAVLink DISTANCE_SENSOR, 30 Hz
                                                        v
                                              +-------------------+
                                              | Flight controller |
                                              | PX4, EKF2         |
                                              +-------------------+</pre>
  <figcaption>
    Figure 1: Downward ToF data flow from the VL53L1X to a subscriber node and the flight
    controller. Arrow rates show the nominal 30 Hz configuration; observed rates depend on
    acquisition and delivery along each path.
  </figcaption>
</figure>

A virtual Duckiedrone supplies simulated measurements to the downstream communication path, using the same ROS 2 message type and topic naming pattern. These can be inspected with the same commands.

## From the ToF Sensor to the Raspberry Pi

The Duckiedrone has five ToF sensors, mounted to look in different directions (all except the back). They share the VL53L1X's I2C address, `0x29`. A [TCA9548A I2C multiplexer](https://www.ti.com/document-viewer/TCA9548A/datasheet) separates their connections into channels, allowing the Raspberry Pi to address each sensor through its configured bus. [Table 1](#table-1) lists the connections.

<table id="table-1" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 1: Duckiedrone ToF sensor connections.</caption>
  <thead>
    <tr><th>Sensor name</th><th>Viewing direction</th><th>Multiplexer channel</th><th>Configured I2C bus</th></tr>
  </thead>
  <tbody>
    <tr><td><code>bottom</code></td><td>Downward</td><td>0</td><td>23</td></tr>
    <tr><td><code>left</code></td><td>Left</td><td>1</td><td>24</td></tr>
    <tr><td><code>right</code></td><td>Right</td><td>2</td><td>25</td></tr>
    <tr><td><code>top</code></td><td>Upward</td><td>3</td><td>26</td></tr>
    <tr><td><code>front</code></td><td>Forward</td><td>4</td><td>27</td></tr>
  </tbody>
</table>

The mounting direction identifies where the sensor looks. Interpreting its range as height requires the sensor's orientation, mounting position, and the geometry of the reflecting surface.

## The drivers

Each sensor has its own driver container running on the Duckiedrone, such as `driver-tof-bottom`. At startup, the physical driver loads its configuration, connects to the sensor, and sets its distance mode and acquisition timing. It also publishes information describing the configured range bounds and field of view.

During operation, the driver waits for a new result and reads the sensor's distance in millimeters. It converts an accepted numerical result to meters and publishes it through [DTPS](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html), together with a timestamp and frame identifier.

The timestamp is assigned after the physical driver retrieves the reading. Differences between successive timestamps therefore describe the spacing of software read times, which includes acquisition and scheduling effects.

The physical driver's frame identifier follows this form, with the final component changing with the sensor name:

```text
DUCKIEDRONE_NAME/tof/bottom
```

There are also two distinct ways that a usable reading can fail to reach a consumer node:

- A published packet can contain no usable distance, for example when a result exceeds the driver's configured upper bound.

- A read error or timeout can prevent the driver from publishing a packet for that iteration.

The first case can become a marker value in ROS 2. The second appears as a gap in the stream. A consumer needs to consider both the value and whether new messages are arriving.

## The ROS 2 bridge

The bridge runs in a separate container, such as `ros2-tof-bottom`. It subscribes to the driver's DTPS information and range data, then publishes ROS 2 messages.

For the bottom sensor, the topic is:

```text
/DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

The [bridge shell opened in Time-of-Flight (ToF) Sensor Fundamentals](./1-tof-basics.ipynb#open-a-shell-inside-the-bridge) inside `ros2-tof-bottom` can be reused to inspect one message. Replace `DUCKIEDRONE_NAME` with the actual Duckiedrone name, and run:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

The command exits after receiving one message, like the one below.

```shell
header:
  stamp:
    sec: 1787340653
    nanosec: 105715314
  frame_id: DUCKIEDRONE_NAME/tof/bottom
radiation_type: 1
field_of_view: 0.471238911151886
min_range: 0.03999999910593033
max_range: 3.5999999046325684
range: 0.017999999225139618
variance: 0.0
```

If it keeps waiting, no message has reached that command yet. Press `Ctrl+C` to stop it.

### What the bridge fills in

[Table 2](#table-2) describes the fields expected from the physical bottom sensor with the current long-range driver profile. The values follow the driver and bridge implementation; device configuration and simulated sources can produce differences.

<table id="table-2" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 2: Interpretation of a bottom-ToF ROS 2 Range message.</caption>
  <thead>
    <tr><th>Field</th><th>Expected content</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>header.stamp</code></td><td>Source timestamp, when available</td><td>The physical driver supplies its read time. The bridge uses its own clock if the source timestamp is missing.</td></tr>
    <tr><td><code>header.frame_id</code></td><td><code>DUCKIEDRONE_NAME/tof/bottom</code></td><td>Identifies the source frame. The bridge forwards the source value; it can be empty when none is supplied.</td></tr>
    <tr><td><code>radiation_type</code></td><td><code>1</code></td><td>The Range message's infrared designation.</td></tr>
    <tr><td><code>field_of_view</code></td><td>Approximately <code>0.471</code> rad</td><td>The driver's advertised 27 deg field of view.</td></tr>
    <tr><td><code>min_range</code></td><td><code>0.04</code> m</td><td>The driver's advertised lower range bound.</td></tr>
    <tr><td><code>max_range</code></td><td><code>3.6</code> m</td><td>The long-range profile's advertised upper bound.</td></tr>
    <tr><td><code>range</code></td><td>Distance in meters, or <code>999.0</code></td><td>The converted reading, or the bridge's unusable-reading marker.</td></tr>
    <tr><td><code>variance</code></td><td><code>0.0</code></td><td>Zero denotes unknown variance; this bridge supplies no measured uncertainty estimate.</td></tr>
  </tbody>
</table>

By Duckietown convention, the bridge emits `999.0` when the incoming distance is missing or nonnumeric, or when it exceeds the advertised maximum. Subscribers should accept only finite ranges within both `min_range` and `max_range`, as required by the Range message contract.


## The other consumer: the flight controller

The bottom sensor also supplies range information for height estimation by the flight controller. MAVROS subscribes to the same ROS 2 range topic and forwards the information in a MAVLink distance-sensor message. The Duckiedrone configuration identifies this sensor as downward-facing.

PX4 can use this distance observation for height estimation, accounting for the sensor's orientation. Tilting the Duckiedrone changes the relationship between viewing direction and vertical distance, and an intervening object can become the observed surface.

The subscriber used in this LX is a separate consumer. The MAVROS branch continues if the analysis subscriber stops. Stopping the shared driver or bridge interrupts new measurements for both branches.


## From the bridge to a subscriber

A subscriber receives messages after the sensor acquisition, driver read, and bridge conversion have occurred. This creates three quantities that should be kept separate:

- __Configured frequency:__ the driver's requested publication rate.

- __Observed arrival rate:__ how often messages reach a particular subscriber.

- __Age of a stored reading:__ how long ago the reading being used was produced or received.

A requested frequency of 30 Hz does not guarantee that every subscriber receives a fresh message every 33 milliseconds. Sensor readiness, read failures, processing, and communication can affect the stream.

The ROS 2 command-line tools provide a first observation of arrival rate. In the bridge shell, run:

```bash
ros2 topic hz /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

Allow several reporting windows to appear, then stop the command with `Ctrl+C`. The reported rate describes messages received by this command during that observation.

A subscriber may retain and reuse its last received value after the stream stops. [Subscribing to the ToF sensor data](./4-subscribing-to-tof.ipynb) introduces a Python subscriber, where distinguishing message arrival from access to a stored value becomes especially important.

## Not all five sensors are treated the same

The default configuration gives the bottom sensor a different role from the other four sensors, as summarized in [Table 3](#table-3).

<table id="table-3" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 3: Default sensor startup and acquisition configuration.</caption>
  <thead>
    <tr><th>Sensor</th><th>Started by default</th><th>Requested frequency</th><th>Timing budget</th></tr>
  </thead>
  <tbody>
    <tr><td><code>bottom</code></td><td>Yes</td><td>30 Hz</td><td>33 ms</td></tr>
    <tr><td><code>left</code></td><td>No</td><td>5 Hz</td><td>200 ms</td></tr>
    <tr><td><code>right</code></td><td>No</td><td>5 Hz</td><td>200 ms</td></tr>
    <tr><td><code>top</code></td><td>No</td><td>5 Hz</td><td>200 ms</td></tr>
    <tr><td><code>front</code></td><td>No</td><td>5 Hz</td><td>200 ms</td></tr>
  </tbody>
</table>

Each active sensor uses a driver container and a bridge container. Activating all five therefore runs ten containers for these two stages. [All Five Sensors](./5-all-five-sensors.ipynb) covers starting and observing the additional sensors.

A sensor that has not been started has no active stream to inspect. Its absence from a topic listing should not immediately be interpreted as a hardware failure.

## Changing any of this

The configuration files on the Duckiedrone determine the physical driver's settings. The device-specific file for a sensor is:

```text
/data/config/node/tof_driver_<sensor_name>/DUCKIEDRONE_NAME.yaml
```

A default file is copied into this location when the device-specific configuration is first created. An existing device-specific file can therefore differ from the defaults in a newer container image.

The following inspection command belongs in the Duckiedrone's device shell, outside the bridge container. If the current shell was opened with `docker exec`, `exit` returns to the device shell:

```bash
cat /data/config/node/tof_driver_bottom/DUCKIEDRONE_NAME.yaml
```

A relevant excerpt of the bottom sensor's default settings is:

```yaml
sensor_name: "bottom"
sensor_model: "VL53L1X"
frequency: 30
mode: "LONG_RANGE"
timing_budget: 33
display_fragment_frequency: 0
connectors:
  - bus: 23
    address: 0x29
```

Here, `frequency` is in hertz and `timing_budget` is in milliseconds. An optional `inter_measurement_period` also uses milliseconds.

When the inter-measurement period is omitted, the driver derives it from the requested frequency and timing budget. The period cannot be shorter than the timing budget, and the driver caps its scheduling frequency accordingly.

For example, consider a side sensor configured for 5 Hz with a 200 ms timing budget:

1. Changing only `frequency` to 10 does not produce a 10 Hz scheduling target. A 200 ms budget still limits the target to 5 Hz.

2. Changing the budget to 100 ms permits a 10 Hz target when the inter-measurement period is omitted.

3. If an explicit inter-measurement period remains at 200 ms, the target remains limited to 5 Hz.

These constraints set the scheduling target. Observe message arrivals and characterize measurement spread under the intended conditions to evaluate the resulting performance.


The file belongs to `root`, so editing it requires `sudo`, for example `sudo nano /data/config/node/tof_driver_bottom/DUCKIEDRONE_NAME.yaml`. Configuration changes are loaded when the driver starts. For a later configuration experiment, preserve the original device settings before editing. Keep a physical Duckiedrone disarmed while changing or restarting this shared measurement path.

After saving a change, restart the driver and then the bridge from the device shell:

```bash
docker restart driver-tof-bottom
docker restart ros2-tof-bottom
```

The driver starts with the new configuration and publishes its sensor information. Restarting the bridge afterward gives it fresh subscriptions and sensor metadata. This order is a practical way to apply the change without relying on reconnection behavior.

To check that measurements have resumed with the new settings, reopen a shell inside the bridge:

```bash
docker exec -it ros2-tof-bottom bash
```

Then repeat the one-message inspection and arrival-rate observation from earlier in this notebook. Check the relevant metadata as well as message arrival. After an experiment, restore the saved configuration, repeat the restart sequence, and verify the restored stream.

Evaluate physical acquisition settings on a physical Duckiedrone. A virtual Duckiedrone receives measurements from the simulated scene through its separate acquisition path.


## Summary and next steps

A ToF reading reaches ROS 2 through a sensor, a physical driver, and a communication bridge. The driver retrieves the distance and supplies its physical read timestamp; the bridge packages the result as a Range message. The same bottom-sensor topic supplies both an analysis subscriber and the flight-controller path.

Interpreting that message requires checking its range bounds and freshness. A numerical value can be outside the advertised interval, and a stored value can remain available after new messages stop arriving. Configured frequency also needs to be distinguished from observed arrival rate.

[Subscribing to the ToF sensor data](./4-subscribing-to-tof.ipynb) uses a Python subscriber to work with this stream.

## Further reading

- [VL53L1X product page](https://www.st.com/en/imaging-and-photonics-solutions/vl53l1x.html): sensor documentation and manufacturer resources.

- [ROS 2 Range message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg): message fields, units, and interpretation.

- [Duckietown physical ToF driver](https://github.com/duckietown/dt-duckiebot-interface/blob/ente/packages/tof_driver/include/tof_driver_node/main.py): configuration, acquisition, timestamps, and DTPS publication.

- [Duckietown ToF ROS 2 bridge](https://github.com/duckietown/dt-ros2-interface/blob/ente/packages/tof_driver/tof_driver/tof_driver_node.py): message conversion and range handling.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()